# 8. Scaled Benchmark: Vector RAG vs. a Graph Agent on MetaQA (1-, 2- and 3-hop)

**Presentation map:** Slide 32 (vector RAG is the baseline — where it breaks), Slide 36 (graphs help on reasoning, not
on fact lookup), Slide 38 (decision rule: single-hop → vector; multi-hop → traversal), Slides 40-41 (traversal lifts
multi-hop KGQA).

Notebooks 1 and 2 showed the pattern on 22 and 10 hand-built questions. This notebook repeats the comparison on a
**standard public benchmark**, with 300 questions and confidence intervals.

**Dataset:** **MetaQA** (Zhang et al., *Variational Reasoning for Question Answering with Knowledge Graph*, AAAI 2018;
[github.com/yuyuz/MetaQA](https://github.com/yuyuz/MetaQA), CC BY-ND 3.0 per the original release). The files are
mirrored on Hugging Face (`camazlucas/MetaQA`) and stored under `data/benchmarks/metaqa/`. Their line counts match the
official release exactly:
- `kb.txt` — the WikiMovies knowledge graph: **134,741 lines (133,582 unique triples), 43,234 entities, 9 relation types**
  (`directed_by`, `written_by`, `starred_actors`, `release_year`, `in_language`, `has_genre`, `has_tags`, …).
- `qa_test_{1,2,3}hop.txt` — the *vanilla* test splits: **9,947 / 14,872 / 14,274** questions. The topic entity is
  marked in `[brackets]` and the gold answer is a set of entities.

We use **test splits only**. The 2- and 3-hop questions are template-generated, and one analysis found about 32% wrong
answers in the 3-hop *training* split. Some label noise in the test splits is therefore likely, and it affects both
methods equally.

**Design.** We sample 100 test questions per hop with a fixed seed and score both methods against the same gold
answer sets, using string matching and no LLM judge:
- **(a) Vector RAG — a strong baseline.** One document per entity lists all of its facts in both directions (e.g.
  *"Movie Kismet: directed by William Dieterle; starring Marlene Dietrich; …"*). The 43,234 documents are embedded with
  `text-embedding-3-small`. For each question, the top 10 documents go to `gpt-4o-mini`.
- **(b) Graph agent.** A `networkx` graph built from `kb.txt`. `gpt-4o-mini` gets a single templated, read-only
  tool, `get_neighbors(entities, relation)`, and runs a ReAct loop with a step budget.

Embeddings and per-question results are cached in `data/benchmarks/metaqa/`. With `RUN_LIVE = False` the notebook
re-runs in seconds; set it to `True` to repeat every API call.

In [1]:
import json, os, random, re, time
from collections import defaultdict, Counter
import networkx as nx

DATA = "data/benchmarks/metaqa"
SEED, N_PER_HOP = 7, 100
RUN_LIVE = False   # True = repeat every API call; missing cache entries are always computed

# --- Knowledge graph: one directed, labelled edge per triple ---
G = nx.MultiDiGraph()
with open(f"{DATA}/kb.txt", encoding="utf-8") as f:
    for line in f:
        h, r, t = line.rstrip("\n").split("|")
        G.add_edge(h, t, key=r, rel=r)
rel_counts = Counter(r for _, _, r in G.edges(keys=True))
print(f"MetaQA KB: {G.number_of_nodes():,} entities, {G.number_of_edges():,} triples, {len(rel_counts)} relations")
print("  " + ", ".join(f"{r} {n:,}" for r, n in rel_counts.most_common()))

# --- Test questions: '[topic entity]' in the text, gold answers separated by '|' ---
def load_split(hop):
    rows = []
    with open(f"{DATA}/qa_test_{hop}hop.txt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            q, a = line.rstrip("\n").split("\t")
            rows.append({"qid": f"{hop}hop-{i}", "hop": hop, "question": q,
                         "topic": re.search(r"\[(.+?)\]", q).group(1), "gold": a.split("|")})
    return rows

splits = {h: load_split(h) for h in (1, 2, 3)}
rng = random.Random(SEED)
questions = [q for h in (1, 2, 3) for q in rng.sample(splits[h], N_PER_HOP)]
print(f"\nTest split sizes: " + ", ".join(f"{h}-hop {len(s):,}" for h, s in splits.items()))
print(f"Sampled {len(questions)} questions ({N_PER_HOP} per hop, seed {SEED}). Examples:")
for h in (1, 2, 3):
    q = next(x for x in questions if x["hop"] == h)
    print(f"  {h}-hop: {q['question']!r} -> {len(q['gold'])} gold answers, e.g. {q['gold'][:3]}")

MetaQA KB: 43,234 entities, 133,582 triples, 9 relations
  starred_actors 33,718, has_tags 28,719, written_by 19,127, release_year 16,715, directed_by 15,957, has_genre 15,485, in_language 3,428, has_imdb_rating 314, has_imdb_votes 119

Test split sizes: 1-hop 9,947, 2-hop 14,872, 3-hop 14,274
Sampled 300 questions (100 per hop, seed 7). Examples:
  1-hop: 'what are some words that describe movie [Blue Like Jazz]' -> 1 gold answers, e.g. ['college']
  2-hop: 'what are the films that are directed by the same director of [Embodiment of Evil]' -> 1 gold answers, e.g. ["At Midnight I'll Take Your Soul"]
  3-hop: 'what genres are the movies written by [A Song Is Born] writers' -> 5 gold answers, e.g. ['Drama', 'Romance', 'Comedy']


## (a) A strong vector baseline: one verbalised document per entity (Slide 32)

Each entity's document lists **all its facts in both directions**, grouped by relation, so a 1-hop question can be
answered from the topic entity's document alone. Hub entities such as *Drama* (4,304 movies) are cut off at about
1,500 characters. The embeddings are cached as `.npy`, so only the first run pays for them.

In [2]:
import numpy as np
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
EMBED_MODEL, CHAT_MODEL = "text-embedding-3-small", "gpt-4o-mini"
PHRASE = {"directed_by": ("directed by", "director of"), "written_by": ("written by", "writer of"),
          "starred_actors": ("starring", "acted in"), "release_year": ("released in", "release year of"),
          "in_language": ("in language", "language of"), "has_genre": ("genre", "genre of"),
          "has_tags": ("tagged", "tag of"), "has_imdb_votes": ("IMDb votes", "IMDb votes of"),
          "has_imdb_rating": ("IMDb rating", "IMDb rating of")}

def entity_doc(e, max_chars=1500):
    groups = defaultdict(list)
    for _, t, r in G.out_edges(e, keys=True):
        groups[PHRASE[r][0]].append(t)
    for s, _, r in G.in_edges(e, keys=True):
        groups[PHRASE[r][1]].append(s)
    body = "; ".join(f"{k}: {', '.join(v)}" for k, v in groups.items())
    return f"{e} — {body}"[:max_chars]

entities = sorted(G.nodes())
docs = [entity_doc(e) for e in entities]

def embed(texts, batch=512):
    out = []
    for i in range(0, len(texts), batch):
        for attempt in range(5):
            try:
                out.extend(d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch]).data)
                break
            except Exception as ex:
                time.sleep(2 ** attempt)
        else:
            raise RuntimeError("embedding failed")
    return np.array(out, dtype=np.float32)

emb_path, names_path = f"{DATA}/entity_emb.npy", f"{DATA}/entity_names.json"
t0 = time.time()
if os.path.exists(emb_path) and json.load(open(names_path)) == entities:
    E = np.load(emb_path)
    print(f"Loaded cached embeddings {E.shape}")
else:
    E = embed(docs)
    np.save(emb_path, E); json.dump(entities, open(names_path, "w"))
    print(f"Embedded {len(docs):,} entity documents in {time.time() - t0:.0f} s")
E /= np.linalg.norm(E, axis=1, keepdims=True)
print(f"Average document length: {np.mean([len(d) for d in docs]):.0f} chars. Example:\n  {docs[entities.index('Kismet')][:300]}")

Loaded cached embeddings (43234, 1536)
Average document length: 121 chars. Example:
  Kismet — directed by: William Dieterle; written by: Edward Knoblock; starring: Marlene Dietrich, Edward Arnold, Ronald Colman, James Craig; released in: 1944; in language: English; tagged: bd-r


In [3]:
# --- Shared helpers: retry, JSON answers, scoring ---
def chat(**kw):
    for attempt in range(6):
        try:
            return client.chat.completions.create(model=CHAT_MODEL, temperature=0, seed=7, **kw)
        except Exception as ex:
            time.sleep(2 ** attempt)
    raise RuntimeError("chat failed")

ANSWER_RULES = ("Return JSON {\"answers\": [...]} listing EVERY entity that answers the question, using the exact "
                "entity names; put the most likely answer first; do not include the topic entity itself unless it is "
                "an answer; return an empty list if you cannot find the answer.")

def parse_answers(text):
    try:
        return [str(a) for a in json.loads(text).get("answers", [])]
    except Exception:
        return []

norm = lambda s: re.sub(r"\s+", " ", s.lower().strip())

def score(pred, gold):
    P, Gs = [norm(p) for p in pred], {norm(g) for g in gold}
    hit1 = bool(P) and P[0] in Gs
    tp = len(set(P) & Gs)
    prec = tp / len(set(P)) if P else 0.0
    rec = tp / len(Gs)
    f1 = 2 * prec * rec / (prec + rec) if tp else 0.0
    return {"hits1": hit1, "f1": f1, "precision": prec, "recall": rec}

# --- (a) Vector RAG: top-10 entity documents -> gpt-4o-mini ---
def vector_rag(q, qvec, k=10):
    top = np.argsort(-(E @ qvec))[:k]
    context = "\n".join(f"[{i + 1}] {docs[j]}" for i, j in enumerate(top))
    r = chat(response_format={"type": "json_object"}, messages=[
        {"role": "system", "content": "Answer movie questions using ONLY the retrieved knowledge-base documents. " + ANSWER_RULES},
        {"role": "user", "content": f"Question: {q['question']}\n\nRetrieved documents:\n{context}"}])
    return {"pred": parse_answers(r.choices[0].message.content), "tokens": r.usage.total_tokens, "tool_calls": 0,
            "retrieved_topic": q["topic"] in {entities[j] for j in top}}

demo = next(x for x in questions if x["hop"] == 2)
demo_vec = embed([demo["question"]])[0]; demo_vec /= np.linalg.norm(demo_vec)
res = vector_rag(demo, demo_vec)
print(f"Q: {demo['question']}\nVector RAG -> {res['pred'][:6]}\nGold       -> {demo['gold'][:6]}\n{score(res['pred'], demo['gold'])}")

Q: what are the films that are directed by the same director of [Embodiment of Evil]
Vector RAG -> ["At Midnight I'll Take Your Soul"]
Gold       -> ["At Midnight I'll Take Your Soul"]
{'hits1': True, 'f1': 1.0, 'precision': 1.0, 'recall': 1.0}


## (b) A graph agent with one templated, read-only tool (Slides 20, 40-43)

The agent sees only `get_neighbors(entities, relation)`, a fixed traversal template over the `networkx` graph, and
never free-form queries. It returns both edge directions, labelled. The tool accepts a *list* of entities, so one
call can expand a whole frontier: a 3-hop question needs about three calls, not one call per movie. Large results are
cut off at 150 names, with the total count reported. The agent has a budget of 8 tool steps.

*Tool design matters (Slides 43-44).* A first version made the agent choose an edge direction. It often asked for
*outgoing* `directed_by` edges from a person, got an empty result and answered "none", which sank 1-hop accuracy to
56%. Returning both directions, labelled, and matching names case-insensitively fixed it — the graph was never the
problem.

In [4]:
MAX_LIST, MAX_STEPS = 150, 8
LOWER = {}
for n in G.nodes():
    LOWER.setdefault(n.lower(), n)

def get_neighbors(entities, relation=None):
    """Read-only traversal template: neighbours of `entities` via `relation` (optional), both directions, labelled."""
    if isinstance(entities, str):
        entities = [entities]
    out, missing = defaultdict(set), 0
    for name in entities:
        e = name if name in G else LOWER.get(name.lower().strip())   # exact name, else case-insensitive match
        if e is None:
            missing += 1; continue
        for _, t, r in G.out_edges(e, keys=True):
            if relation in (None, r): out[f"{r} (outgoing: the movie's {r.replace('_', ' ')} value)"].add(t)
        for s, _, r in G.in_edges(e, keys=True):
            if relation in (None, r): out[f"{r} (incoming: movies with this {r.replace('_', ' ')} value)"].add(s)
    result = {k: {"count": len(v), "entities": sorted(v)[:MAX_LIST]} for k, v in out.items()}
    if missing:
        result["not_found"] = missing
    if not out:
        result["note"] = "no edges with this relation for these entities; try another relation or omit it"
    return json.dumps(result, ensure_ascii=False)

TOOLS = [{"type": "function", "function": {
    "name": "get_neighbors",
    "description": ("Traverse the movie knowledge graph. Movies are linked to their directors, writers, actors, release "
                    "year, language, genres and tags (relations: directed_by, written_by, starred_actors, release_year, "
                    "in_language, has_genre, has_tags, has_imdb_rating, has_imdb_votes). Returns neighbours in both "
                    "directions, labelled outgoing (movie -> value) or incoming (value -> movies). Pass several "
                    "entities at once to expand a whole set."),
    "parameters": {"type": "object", "properties": {
        "entities": {"type": "array", "items": {"type": "string"}},
        "relation": {"type": "string", "enum": sorted(rel_counts)}},
        "required": ["entities"]}}}]

def graph_agent(q, trace=False):
    msgs = [{"role": "system", "content": "You answer movie questions by traversing a knowledge graph with the "
             "get_neighbors tool, one hop at a time, starting from the topic entity in [brackets]. " + ANSWER_RULES},
            {"role": "user", "content": q["question"]}]
    tokens = calls = 0
    for step in range(MAX_STEPS + 1):
        kw = {"tools": TOOLS} if step < MAX_STEPS else {}
        r = chat(messages=msgs, response_format={"type": "json_object"}, **kw)
        tokens += r.usage.total_tokens
        m = r.choices[0].message
        if not m.tool_calls:
            return {"pred": parse_answers(m.content or "{}"), "tokens": tokens, "tool_calls": calls}
        msgs.append(m.model_dump(exclude_none=True))
        for tc in m.tool_calls:
            calls += 1
            try:
                args = json.loads(tc.function.arguments)
                obs = get_neighbors(**args)
            except Exception as ex:
                args, obs = {}, json.dumps({"error": str(ex)})
            if trace:
                print(f"  [step {step + 1}] get_neighbors({args}) -> {obs[:160]}…")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": obs})
    return {"pred": [], "tokens": tokens, "tool_calls": calls}

demo3 = next(x for x in questions if x["hop"] == 3)
print(f"Q: {demo3['question']}")
res = graph_agent(demo3, trace=True)
print(f"Graph agent -> {res['pred'][:6]}  ({len(res['pred'])} answers)\nGold        -> {demo3['gold'][:6]}  "
      f"({len(demo3['gold'])} answers)\n{score(res['pred'], demo3['gold'])}")

Q: what genres are the movies written by [A Song Is Born] writers


  [step 1] get_neighbors({'entities': ['A Song Is Born'], 'relation': 'written_by'}) -> {"written_by (outgoing: the movie's written by value)": {"count": 1, "entities": ["Billy Wilder"]}}…


  [step 2] get_neighbors({'entities': ['Billy Wilder'], 'relation': 'has_genre'}) -> {"note": "no edges with this relation for these entities; try another relation or omit it"}…


  [step 3] get_neighbors({'entities': ['Billy Wilder'], 'relation': 'directed_by'}) -> {"directed_by (incoming: movies with this directed by value)": {"count": 23, "entities": ["A Foreign Affair", "Ace in the Hole", "Avanti!", "Buddy Buddy", "Doub…


  [step 4] get_neighbors({'entities': ['A Foreign Affair', 'Ace in the Hole', 'Avanti!', 'Buddy Buddy', 'Double Indemnity', 'Five Graves to Cairo', 'Irma la Douce', 'Kiss Me, Stupid', 'Love in the Afternoon', 'One, Two, Three', 'Sabrina', 'Some Like It Hot', 'Stalag 17', 'The Apartment', 'The Emperor Waltz', 'The Fortune Cookie', 'The Front Page', 'The Lost Weekend', 'The Major and the Minor', 'The Private Life of Sherlock Holmes', 'The Seven Year Itch', 'The Spirit of St. Louis', 'Witness for the Prosecution'], 'relation': 'has_genre'}) -> {"has_genre (outgoing: the movie's has genre value)": {"count": 4, "entities": ["Comedy", "Drama", "Musical", "War"]}}…


Graph agent -> ['Comedy', 'Drama', 'Musical', 'War']  (4 answers)
Gold        -> ['Drama', 'Romance', 'Comedy', 'Musical', 'War']  (5 answers)
{'hits1': True, 'f1': 0.888888888888889, 'precision': 1.0, 'recall': 0.8}


## Run the benchmark: 300 questions × 2 methods

Both methods run in parallel threads with retry and backoff. Results are cached per question in
`results_cache.json`, so a live talk loads them instantly.

In [5]:
cache_path = f"{DATA}/results_cache.json"
cache = {} if RUN_LIVE or not os.path.exists(cache_path) else json.load(open(cache_path))
todo = {m: [q for q in questions if f"{m}|{q['qid']}" not in cache] for m in ("vector", "graph")}
print(f"Cached: vector {len(questions) - len(todo['vector'])}/{len(questions)}, "
      f"graph {len(questions) - len(todo['graph'])}/{len(questions)}; running the rest live")

if todo["vector"] or todo["graph"]:
    t0 = time.time()
    qvecs = {}
    if todo["vector"]:
        V = embed([q["question"] for q in todo["vector"]]); V /= np.linalg.norm(V, axis=1, keepdims=True)
        qvecs = {q["qid"]: v for q, v in zip(todo["vector"], V)}
    jobs = [("vector", q) for q in todo["vector"]] + [("graph", q) for q in todo["graph"]]
    def run_one(job):
        m, q = job
        return m, q["qid"], (vector_rag(q, qvecs[q["qid"]]) if m == "vector" else graph_agent(q))
    with ThreadPoolExecutor(max_workers=8) as pool:
        for n, (m, qid, r_) in enumerate(pool.map(run_one, jobs), 1):
            cache[f"{m}|{qid}"] = r_
            if n % 100 == 0:
                print(f"  {n}/{len(jobs)} done ({time.time() - t0:.0f} s)")
                json.dump(cache, open(cache_path, "w"))
    json.dump(cache, open(cache_path, "w"))
    print(f"Live run finished in {time.time() - t0:.0f} s")

rows = []
for q in questions:
    for method in ("vector", "graph"):
        r = cache[f"{method}|{q['qid']}"]
        rows.append({"hop": q["hop"], "method": method, **score(r["pred"], q["gold"]),
                     "tokens": r["tokens"], "tool_calls": r["tool_calls"], "n_gold": len(q["gold"])})
print(f"Scored {len(rows)} (question, method) pairs")

Cached: vector 300/300, graph 300/300; running the rest live
Scored 600 (question, method) pairs


In [6]:
import math

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def boot_ci(xs, B=2000, seed=SEED):
    r = random.Random(seed); n = len(xs)
    means = sorted(sum(r.choice(xs) for _ in range(n)) / n for _ in range(B))
    return means[int(0.025 * B)], means[int(0.975 * B)]

summary = {}
print(f"{'hop':<6}{'method':<8}{'Hits@1 [95% CI]':<24}{'F1 [95% CI]':<22}{'tokens/q':>9}{'tool calls/q':>14}")
for hop in (1, 2, 3):
    for method in ("vector", "graph"):
        R = [r for r in rows if r["hop"] == hop and r["method"] == method]
        k, n = sum(r["hits1"] for r in R), len(R)
        lo, hi = wilson(k, n)
        f1s = [r["f1"] for r in R]; flo, fhi = boot_ci(f1s)
        s = {"hits1": k / n, "hits1_ci": (lo, hi), "f1": sum(f1s) / n, "f1_ci": (flo, fhi),
             "tokens": sum(r["tokens"] for r in R) / n, "tool_calls": sum(r["tool_calls"] for r in R) / n}
        summary[(hop, method)] = s
        print(f"{hop}-hop {method:<8}{s['hits1']:>5.0%} [{lo:.0%}–{hi:.0%}]{'':<9}{s['f1']:>5.2f} [{flo:.2f}–{fhi:.2f}]"
              f"{'':<6}{s['tokens']:>9,.0f}{s['tool_calls']:>14.1f}")
    print()

hop   method  Hits@1 [95% CI]         F1 [95% CI]            tokens/q  tool calls/q
1-hop vector    91% [84%–95%]          0.91 [0.86–0.95]            500           0.0
1-hop graph     99% [95%–100%]          0.97 [0.94–0.99]            826           1.3

2-hop vector    39% [30%–49%]          0.33 [0.26–0.42]            470           0.0
2-hop graph     97% [92%–99%]          0.98 [0.96–1.00]          1,386           3.1

3-hop vector    36% [27%–46%]          0.18 [0.14–0.23]            624           0.0
3-hop graph     80% [71%–87%]          0.71 [0.64–0.79]          4,129          12.6



## Reading the result

- **1-hop:** both methods do well; the vector baseline finds the topic entity's document, which holds the answer.
- **2- and 3-hop:** the answer sits on entities two or three edges away, whose documents look nothing like the
  question. Top-10 retrieval rarely contains them, while traversal reaches them by construction (Slide 32).
- **Cost:** the graph agent spends more tokens and tool calls as hops grow, mainly to expand large frontiers.
- **Label noise:** MetaQA's 2- and 3-hop answers are template-generated, so some "misses" are gold errors.
  This affects both methods equally.

In [7]:
tot_tokens = sum(r["tokens"] for r in rows)
print("=" * 86)
print("RESULTS SUMMARY — Notebook 8: MetaQA test (100 questions per hop, seed 7, gpt-4o-mini, k=10)")
print("=" * 86)
print(f"KB: {G.number_of_nodes():,} entities / {G.number_of_edges():,} triples; questions sampled from "
      f"{sum(len(s) for s in splits.values()):,} test questions")
print(f"{'':10}{'Vector RAG Hits@1':>22}{'Graph agent Hits@1':>22}{'gap':>8}{'Vector F1':>11}{'Graph F1':>10}")
for hop in (1, 2, 3):
    v, g = summary[(hop, "vector")], summary[(hop, "graph")]
    print(f"{hop}-hop{'':5}{v['hits1']:>8.0%} [{v['hits1_ci'][0]:.0%}–{v['hits1_ci'][1]:.0%}]{'':3}"
          f"{g['hits1']:>8.0%} [{g['hits1_ci'][0]:.0%}–{g['hits1_ci'][1]:.0%}]{'':3}"
          f"{(g['hits1'] - v['hits1']) * 100:>+6.0f} pt{v['f1']:>9.2f}{g['f1']:>10.2f}")
for hop in (1, 2, 3):
    v, g = summary[(hop, "vector")], summary[(hop, "graph")]
    print(f"Cost {hop}-hop: tokens/q vector {v['tokens']:,.0f} vs graph {g['tokens']:,.0f} "
          f"({g['tokens'] / v['tokens']:.1f}x); graph tool calls/q {g['tool_calls']:.1f}")
print(f"Total chat tokens in this benchmark (both methods): {tot_tokens:,}")
print("\nLecture message (Slides 32, 36, 38, 41): the gap grows with the number of hops. On 1-hop lookups a good")
print("vector index is close to the graph; at 2 and 3 hops the answer is spread over entities that do not resemble")
print("the question, top-k retrieval collapses, and traversal holds up — at several times the tokens.")
print("Caveat: MetaQA questions are pure KG lookups and the vector documents are the KG verbalised, so this favours the")
print("graph more than mixed text corpora would (compare Notebook 1, where vector RAG tied on lookups).")

RESULTS SUMMARY — Notebook 8: MetaQA test (100 questions per hop, seed 7, gpt-4o-mini, k=10)
KB: 43,234 entities / 133,582 triples; questions sampled from 39,093 test questions
               Vector RAG Hits@1    Graph agent Hits@1     gap  Vector F1  Graph F1
1-hop          91% [84%–95%]        99% [95%–100%]       +8 pt     0.91      0.97
2-hop          39% [30%–49%]        97% [92%–99%]      +58 pt     0.33      0.98
3-hop          36% [27%–46%]        80% [71%–87%]      +44 pt     0.18      0.71
Cost 1-hop: tokens/q vector 500 vs graph 826 (1.7x); graph tool calls/q 1.3
Cost 2-hop: tokens/q vector 470 vs graph 1,386 (3.0x); graph tool calls/q 3.1
Cost 3-hop: tokens/q vector 624 vs graph 4,129 (6.6x); graph tool calls/q 12.6
Total chat tokens in this benchmark (both methods): 793,504

Lecture message (Slides 32, 36, 38, 41): the gap grows with the number of hops. On 1-hop lookups a good
vector index is close to the graph; at 2 and 3 hops the answer is spread over entities that do no